# Visualización de Métricas de Evaluación XAI

Histogramas de distribución y gráficas de barras por algoritmo:
- **AggDiv** — Diversidad Agregada (nivel usuario) → histograma
- **IXD** — Inter-eXplanation Diversity (nivel usuario) → histograma
- **MIL** — Mean Inter-List Diversity (nivel sistema) → barras
- **ECS** — Explanation Consistency Score (nivel hotel_recomendado) → barras + scatter

**Estructura esperada de ficheros:**
```
output/metricas_evaluacion_{modo}/
    evaluacion_{algoritmo}_AggDiv_{timestamp}.csv
    evaluacion_{algoritmo}_IXD_{timestamp}.csv
    evaluacion_{algoritmo}_MIL_{timestamp}.csv
    evaluacion_{algoritmo}_ECS_{timestamp}.csv
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from pathlib import Path
from collections import defaultdict

# ── Configuración estética ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor':  'white',
    'axes.facecolor':    '#f9f9f7',
    'axes.grid':         True,
    'grid.color':        'white',
    'grid.linewidth':    1.2,
    'axes.spines.top':   False,
    'axes.spines.right': False,
    'font.family':       'DejaVu Sans',
    'font.size':         11,
})

In [ ]:
# ── CONFIGURACIÓN — cambia aquí el modo ──────────────────────────────────────
MODO = 'semi'   # opciones: 'muestra' | 'semi' | 'completo'

# ── Rutas ────────────────────────────────────────────────────────────────────
PROJECT_ROOT = Path().resolve().parent.parent
EVAL_DIR   = PROJECT_ROOT / 'output' / f'metricas_evaluacion_{MODO}'
OUTPUT_DIR = PROJECT_ROOT / 'output' / f'visualizacion_{MODO}'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Leyendo CSVs desde: {EVAL_DIR}')
print(f'Guardando figuras en: {OUTPUT_DIR}')

In [ ]:
# ── Cargar todos los CSVs y agrupar por métrica ──────────────────────────────
#
# datos[metrica][algoritmo] = pd.DataFrame

METRICAS_USUARIO = ['AggDiv', 'IXD']   # granularidad usuario
METRICAS_SISTEMA = ['MIL']             # granularidad sistema (1 fila/algoritmo)
METRICAS_HOTEL   = ['ECS']             # granularidad hotel_recomendado
TODAS_METRICAS   = METRICAS_USUARIO + METRICAS_SISTEMA + METRICAS_HOTEL

datos: dict = defaultdict(dict)

for csv_path in sorted(EVAL_DIR.glob('evaluacion_*.csv')):
    stem = csv_path.stem
    metrica = None
    for m in TODAS_METRICAS:
        if f'_{m}_' in stem:
            metrica = m
            break
    if metrica is None:
        print(f'  ⚠️  No se reconoce métrica en: {csv_path.name}')
        continue

    prefijo   = 'evaluacion_'
    sufijo    = f'_{metrica}_'
    algoritmo = stem[len(prefijo) : stem.index(sufijo)]

    df = pd.read_csv(csv_path)
    datos[metrica][algoritmo] = df
    print(f'  ✅  {metrica:8s}  {algoritmo:40s}  ({len(df)} filas)')

print(f'\nMétricas detectadas : {list(datos.keys())}')
for m, algs in datos.items():
    print(f'  {m}: {list(algs.keys())}')

In [ ]:
# ── Paleta de colores por algoritmo ─────────────────────────────────────────
COLORES_KG = ['#003f9e', '#0077ff', '#00c2c7', '#00875a']
COLORES_CF = ['#cc0000', '#ff6600', '#e6007e', '#ffcc00']

def asignar_colores(algoritmos: list) -> dict:
    colores = {}
    idx_kg, idx_cf = 0, 0
    for alg in sorted(algoritmos):
        if alg.startswith('kg_'):
            colores[alg] = COLORES_KG[idx_kg % len(COLORES_KG)]
            idx_kg += 1
        else:
            colores[alg] = COLORES_CF[idx_cf % len(COLORES_CF)]
            idx_cf += 1
    return colores

todos_algoritmos = set()
for algs in datos.values():
    todos_algoritmos.update(algs.keys())

COLORES = asignar_colores(todos_algoritmos)
print('Colores asignados:')
for alg, col in COLORES.items():
    print(f'  {alg:40s}  {col}')

In [ ]:
# ── Función principal: histograma superpuesto (métricas usuario) ─────────────

def plot_histograma_metrica(
    metrica, columna, datos_algs, colores,
    titulo=None, xlabel=None, bins=15, kde=True, guardar=None,
):
    from scipy.stats import gaussian_kde

    fig, ax = plt.subplots(figsize=(10, 5))

    todos_vals = []
    for df in datos_algs.values():
        if columna in df.columns:
            todos_vals.extend(df[columna].dropna().tolist())
    if not todos_vals:
        print(f'  ⚠️  Sin datos para {metrica} / {columna}')
        return

    x_min, x_max = min(todos_vals), max(todos_vals)
    rango  = x_max - x_min if x_max > x_min else 1
    x_min -= rango * 0.05
    x_max += rango * 0.05

    handles = []
    for algoritmo in sorted(datos_algs.keys()):
        df = datos_algs[algoritmo]
        if columna not in df.columns:
            continue
        vals = df[columna].dropna().values
        if len(vals) == 0:
            continue
        color = colores.get(algoritmo, '#888')
        label = algoritmo.replace('kg_', 'KG: ').replace('cf_', 'CF: ')

        ax.hist(vals, bins=bins, range=(x_min, x_max),
                color=color, alpha=0.30, edgecolor='white', linewidth=0.5)

        if kde and len(np.unique(vals)) > 1:
            kde_f = gaussian_kde(vals, bw_method='scott')
            x_kde = np.linspace(x_min, x_max, 300)
            y_kde = kde_f(x_kde) * len(vals) * (x_max - x_min) / bins
            ax.plot(x_kde, y_kde, color=color, linewidth=2)

        ax.axvline(np.mean(vals), color=color, linewidth=1.1, linestyle='--', alpha=0.75)
        handles.append(mpatches.Patch(color=color, alpha=0.75, label=f'{label}  (μ={np.mean(vals):.3f})'))

    ax.set_xlabel(xlabel or columna, fontsize=11)
    ax.set_ylabel('# usuarios', fontsize=11)
    ax.set_title(titulo or f'{metrica} — {columna}', fontsize=12, fontweight='bold')
    ax.legend(handles=handles, fontsize=8, framealpha=0.85,
              ncol=2 if len(handles) > 4 else 1, loc='upper right')
    ax.set_xlim(x_min, x_max)
    plt.tight_layout()

    if guardar:
        fig.savefig(guardar, dpi=150, bbox_inches='tight')
        print(f'  💾 Guardado: {guardar.name}')
    plt.show()
    plt.close(fig)

## AggDiv — Diversidad Agregada de Explicadores

Número de hoteles explicadores únicos que aparecen en el conjunto de recomendaciones de un usuario.  
Cuanto mayor, más diversa es la cobertura explicativa.

In [ ]:
if 'AggDiv' in datos:
    ejemplo_df  = next(iter(datos['AggDiv'].values()))
    cols_aggdiv = [c for c in ejemplo_df.columns if c.startswith('AggDiv') and 'norm' not in c]
    print(f'Columnas AggDiv disponibles: {cols_aggdiv}')

    for col in cols_aggdiv:
        plot_histograma_metrica(
            metrica    = 'AggDiv',
            columna    = col,
            datos_algs = datos['AggDiv'],
            colores    = COLORES,
            titulo     = f'AggDiv — Distribución de {col} por algoritmo',
            xlabel     = f'{col}  (nº explicadores únicos)',
            bins       = 12,
            kde        = True,
            guardar    = OUTPUT_DIR / f'hist_AggDiv_{col}.png',
        )
else:
    print('No hay datos de AggDiv cargados.')

## IXD — Inter-eXplanation Diversity

Diversidad entre las listas de explicadores de distintas recomendaciones del mismo usuario.  
Rango [0, 1]. NaN si el usuario solo tiene una recomendación.

In [ ]:
if 'IXD' in datos:
    ejemplo_df = next(iter(datos['IXD'].values()))
    cols_ixd   = [c for c in ejemplo_df.columns if c.startswith('IXD')]
    print(f'Columnas IXD disponibles: {cols_ixd}')

    for col in cols_ixd:
        plot_histograma_metrica(
            metrica    = 'IXD',
            columna    = col,
            datos_algs = datos['IXD'],
            colores    = COLORES,
            titulo     = f'IXD — Distribución de {col} por algoritmo',
            xlabel     = f'{col}  (0 = mismos explicadores, 1 = totalmente distintos)',
            bins       = 12,
            kde        = True,
            guardar    = OUTPUT_DIR / f'hist_IXD_{col}.png',
        )
else:
    print('No hay datos de IXD cargados.')

## MIL — Mean Inter-List Diversity

Métrica de sistema: un único valor por algoritmo.  
Mide cuánto difieren los conjuntos de explicadores entre usuarios distintos.  
Rango [0, 1]. 1 = máxima personalización.

In [ ]:
if 'MIL' in datos:
    filas = []
    for algoritmo, df in datos['MIL'].items():
        fila = {'algoritmo': algoritmo}
        for col in [c for c in df.columns if c.startswith('MIL')]:
            fila[col] = df[col].iloc[0]
        filas.append(fila)

    df_mil = pd.DataFrame(filas).set_index('algoritmo')
    print(df_mil.to_string())

    for col in [c for c in df_mil.columns if c.startswith('MIL')]:
        fig, ax = plt.subplots(figsize=(9, max(3, len(df_mil) * 0.55)))
        algoritmos = df_mil.index.tolist()
        valores    = df_mil[col].values
        colores_barras = [COLORES.get(a, '#888') for a in algoritmos]

        bars = ax.barh(algoritmos, valores, color=colores_barras, alpha=0.8, height=0.5)
        ax.set_xlabel(f'{col}  (0 = sin personalización, 1 = totalmente personalizado)', fontsize=11)
        ax.set_title(f'MIL — {col} por algoritmo (métrica de sistema)', fontsize=12, fontweight='bold')
        ax.set_xlim(0, 1.05)
        ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9)
        ax.invert_yaxis()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        ruta = OUTPUT_DIR / f'barras_MIL_{col}.png'
        fig.savefig(ruta, dpi=150, bbox_inches='tight')
        print(f'  💾 Guardado: {ruta.name}')
        plt.show()
        plt.close(fig)
else:
    print('No hay datos de MIL cargados.')

## ECS — Explanation Consistency Score

Mide el solapamiento (Jaccard) entre los conjuntos de explicadores que distintos usuarios
recibieron para el **mismo hotel recomendado**.  
Rango [0, 1]. 1 = todos los usuarios reciben exactamente los mismos explicadores para ese hotel.

Se visualizan:
1. **Barras** — ECS medio por algoritmo (media sobre todos los `hotel_recomendado`)
2. **Histograma** — distribución de ECS por hotel para cada algoritmo
3. **Scatter** — ECS vs nº usuarios que recibieron ese hotel (¿el solapamiento cae con popularidad?)

In [ ]:
if 'ECS' in datos:
    # ── 1. Tabla resumen: ECS medio por algoritmo y @k ───────────────────────
    filas = []
    for algoritmo, df in datos['ECS'].items():
        fila = {'algoritmo': algoritmo}
        for col in [c for c in df.columns if c.startswith('ECS')]:
            fila[col] = round(df[col].dropna().mean(), 6)
        fila['n_hoteles'] = len(df)
        filas.append(fila)

    df_ecs_resumen = pd.DataFrame(filas).set_index('algoritmo')
    print('ECS medio por algoritmo:')
    print(df_ecs_resumen.to_string())
else:
    print('No hay datos de ECS cargados.')

In [ ]:
# ── 2. Barras: ECS medio por algoritmo para cada variante @k ─────────────────
if 'ECS' in datos:
    cols_ecs = [c for c in df_ecs_resumen.columns if c.startswith('ECS')]

    for col in cols_ecs:
        fig, ax = plt.subplots(figsize=(9, max(3, len(df_ecs_resumen) * 0.55)))
        algoritmos     = df_ecs_resumen.index.tolist()
        valores        = df_ecs_resumen[col].values
        colores_barras = [COLORES.get(a, '#888') for a in algoritmos]

        bars = ax.barh(algoritmos, valores, color=colores_barras, alpha=0.8, height=0.5)
        ax.set_xlabel(
            f'{col}  (0 = explicadores completamente distintos por usuario,\n'
            '1 = todos los usuarios reciben los mismos explicadores)',
            fontsize=10
        )
        ax.set_title(
            f'ECS medio — {col} por algoritmo\n'
            f'(media sobre todos los hoteles recomendados con ≥2 usuarios)',
            fontsize=12, fontweight='bold'
        )
        ax.set_xlim(0, max(valores.max() * 1.15, 0.05))
        ax.bar_label(bars, fmt='%.4f', padding=4, fontsize=9)
        ax.invert_yaxis()
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

        plt.tight_layout()
        ruta = OUTPUT_DIR / f'barras_ECS_{col}.png'
        fig.savefig(ruta, dpi=150, bbox_inches='tight')
        print(f'  💾 Guardado: {ruta.name}')
        plt.show()
        plt.close(fig)

In [ ]:
# ── 3. Histograma: distribución de ECS por hotel_recomendado ─────────────────
if 'ECS' in datos:
    for col in ['ECS', 'ECS@1', 'ECS@3', 'ECS@5']:
        fig, ax = plt.subplots(figsize=(10, 5))
        handles = []

        todos_vals = []
        for df in datos['ECS'].values():
            if col in df.columns:
                todos_vals.extend(df[col].dropna().tolist())
        if not todos_vals:
            plt.close(fig)
            continue

        x_min, x_max = 0.0, 1.0

        for algoritmo in sorted(datos['ECS'].keys()):
            df = datos['ECS'][algoritmo]
            if col not in df.columns:
                continue
            vals = df[col].dropna().values
            if len(vals) == 0:
                continue
            color = COLORES.get(algoritmo, '#888')
            label = algoritmo.replace('kg_', 'KG: ').replace('cf_', 'CF: ')

            ax.hist(vals, bins=20, range=(x_min, x_max),
                    color=color, alpha=0.35, edgecolor='white', linewidth=0.5)
            ax.axvline(np.mean(vals), color=color, linewidth=1.2, linestyle='--', alpha=0.8)
            handles.append(mpatches.Patch(
                color=color, alpha=0.8,
                label=f'{label}  (μ={np.mean(vals):.3f}, n={len(vals)} hoteles)'
            ))

        ax.set_xlabel(f'{col}  (Jaccard medio entre usuarios para el mismo hotel)', fontsize=11)
        ax.set_ylabel('# hoteles recomendados', fontsize=11)
        ax.set_title(
            f'ECS — Distribución de {col} por hotel_recomendado y algoritmo',
            fontsize=12, fontweight='bold'
        )
        ax.legend(handles=handles, fontsize=8, framealpha=0.85,
                  ncol=2 if len(handles) > 4 else 1)
        ax.set_xlim(x_min, x_max)
        plt.tight_layout()

        ruta = OUTPUT_DIR / f'hist_ECS_{col}.png'
        fig.savefig(ruta, dpi=150, bbox_inches='tight')
        print(f'  💾 Guardado: {ruta.name}')
        plt.show()
        plt.close(fig)

In [ ]:
# ── 4. Scatter: ECS vs popularidad del hotel (n_usuarios) ────────────────────
#
# Hipótesis: hoteles muy populares (muchos usuarios) tienden a tener
# explicadores más heterogéneos → ECS más bajo.

if 'ECS' in datos:
    fig, ax = plt.subplots(figsize=(10, 6))
    handles = []

    for algoritmo in sorted(datos['ECS'].keys()):
        df = datos['ECS'][algoritmo]
        if 'ECS' not in df.columns or 'n_usuarios' not in df.columns:
            continue
        sub = df[['n_usuarios', 'ECS']].dropna()
        if sub.empty:
            continue
        color = COLORES.get(algoritmo, '#888')
        label = algoritmo.replace('kg_', 'KG: ').replace('cf_', 'CF: ')

        ax.scatter(
            sub['n_usuarios'], sub['ECS'],
            color=color, alpha=0.45, s=18, edgecolors='none'
        )
        # Línea de tendencia
        if len(sub) > 2:
            z = np.polyfit(np.log1p(sub['n_usuarios']), sub['ECS'], 1)
            p = np.poly1d(z)
            x_sorted = np.sort(sub['n_usuarios'].values)
            ax.plot(x_sorted, p(np.log1p(x_sorted)), color=color, linewidth=1.5, alpha=0.85)
        handles.append(mpatches.Patch(color=color, alpha=0.8, label=label))

    ax.set_xlabel('Nº de usuarios que recibieron el hotel_recomendado (popularidad)', fontsize=11)
    ax.set_ylabel('ECS  (Jaccard medio de explicadores)', fontsize=11)
    ax.set_title(
        'ECS vs Popularidad del hotel recomendado\n'
        '(¿los hoteles populares tienen explicaciones menos consistentes?)',
        fontsize=12, fontweight='bold'
    )
    ax.legend(handles=handles, fontsize=9, framealpha=0.85,
              ncol=2 if len(handles) > 4 else 1)
    ax.set_ylim(-0.02, 1.02)
    plt.tight_layout()

    ruta = OUTPUT_DIR / 'scatter_ECS_vs_popularidad.png'
    fig.savefig(ruta, dpi=150, bbox_inches='tight')
    print(f'  💾 Guardado: {ruta.name}')
    plt.show()
    plt.close(fig)

## Vista global — AggDiv, IXD, MIL y ECS en un panel 2×2

Comparativa rápida de las cuatro métricas en un único vistazo.

In [ ]:
from scipy.stats import gaussian_kde

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# ── Paneles 1 y 2: histogramas AggDiv e IXD ──────────────────────────────────
for (metrica, columna, ax, xlabel) in [
    ('AggDiv', 'AggDiv', axes[0, 0], 'AggDiv  (nº explicadores únicos)'),
    ('IXD',    'IXD',    axes[0, 1], 'IXD  (0 = iguales, 1 = distintos)'),
]:
    if metrica not in datos:
        ax.set_title(f'{metrica}: sin datos')
        continue

    todos_vals = []
    for df in datos[metrica].values():
        if columna in df.columns:
            todos_vals.extend(df[columna].dropna().tolist())
    if not todos_vals:
        ax.set_title(f'{columna}: sin datos')
        continue

    x_min, x_max = min(todos_vals), max(todos_vals)
    rango  = x_max - x_min if x_max > x_min else 1
    x_min -= rango * 0.05
    x_max += rango * 0.05

    handles = []
    for algoritmo in sorted(datos[metrica].keys()):
        df = datos[metrica][algoritmo]
        if columna not in df.columns:
            continue
        vals = df[columna].dropna().values
        if len(vals) == 0:
            continue
        color = COLORES.get(algoritmo, '#888')
        label = algoritmo.replace('kg_', 'KG: ').replace('cf_', 'CF: ')
        ax.hist(vals, bins=12, range=(x_min, x_max),
                color=color, alpha=0.28, edgecolor='white', linewidth=0.5)
        if len(np.unique(vals)) > 1:
            kde_f = gaussian_kde(vals, bw_method='scott')
            x_kde = np.linspace(x_min, x_max, 300)
            y_kde = kde_f(x_kde) * len(vals) * (x_max - x_min) / 12
            ax.plot(x_kde, y_kde, color=color, linewidth=2)
        ax.axvline(np.mean(vals), color=color, linewidth=1.1, linestyle='--', alpha=0.75)
        handles.append(mpatches.Patch(color=color, alpha=0.75, label=label))

    ax.set_xlabel(xlabel, fontsize=10)
    ax.set_ylabel('# usuarios', fontsize=10)
    ax.set_title(columna, fontsize=12, fontweight='bold')
    ax.legend(handles=handles, fontsize=7, framealpha=0.85,
              ncol=2 if len(handles) > 4 else 1)
    ax.set_xlim(x_min, x_max)
    ax.set_facecolor('#f9f9f7')
    ax.grid(True, color='white', linewidth=1.1)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# ── Panel 3: barras MIL ───────────────────────────────────────────────────────
ax_mil = axes[1, 0]
if 'MIL' in datos:
    mil_vals = {alg: df['MIL'].iloc[0] for alg, df in datos['MIL'].items() if 'MIL' in df.columns}
    algs_sorted = sorted(mil_vals.keys())
    vals_sorted = [mil_vals[a] for a in algs_sorted]
    cols_bars   = [COLORES.get(a, '#888') for a in algs_sorted]
    bars = ax_mil.barh(algs_sorted, vals_sorted, color=cols_bars, alpha=0.8, height=0.5)
    ax_mil.set_xlabel('MIL  (diversidad entre usuarios)', fontsize=10)
    ax_mil.set_title('MIL — métrica de sistema', fontsize=12, fontweight='bold')
    ax_mil.set_xlim(0, 1.05)
    ax_mil.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
    ax_mil.invert_yaxis()
else:
    ax_mil.set_title('MIL: sin datos')
ax_mil.set_facecolor('#f9f9f7')
ax_mil.spines['top'].set_visible(False)
ax_mil.spines['right'].set_visible(False)

# ── Panel 4: barras ECS medio ─────────────────────────────────────────────────
ax_ecs = axes[1, 1]
if 'ECS' in datos:
    ecs_vals = {}
    for alg, df in datos['ECS'].items():
        if 'ECS' in df.columns:
            ecs_vals[alg] = df['ECS'].dropna().mean()
    algs_sorted = sorted(ecs_vals.keys())
    vals_sorted = [ecs_vals[a] for a in algs_sorted]
    cols_bars   = [COLORES.get(a, '#888') for a in algs_sorted]
    bars = ax_ecs.barh(algs_sorted, vals_sorted, color=cols_bars, alpha=0.8, height=0.5)
    ax_ecs.set_xlabel('ECS medio  (consistencia de explicadores por hotel)', fontsize=10)
    ax_ecs.set_title('ECS — métrica de hotel recomendado', fontsize=12, fontweight='bold')
    vmax = max(vals_sorted) * 1.2 if vals_sorted else 0.5
    ax_ecs.set_xlim(0, max(vmax, 0.05))
    ax_ecs.bar_label(bars, fmt='%.4f', padding=3, fontsize=8)
    ax_ecs.invert_yaxis()
else:
    ax_ecs.set_title('ECS: sin datos')
ax_ecs.set_facecolor('#f9f9f7')
ax_ecs.spines['top'].set_visible(False)
ax_ecs.spines['right'].set_visible(False)

fig.suptitle('Métricas de evaluación XAI — resumen por algoritmo', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
ruta_global = OUTPUT_DIR / 'panel_global_metricas.png'
fig.savefig(ruta_global, dpi=150, bbox_inches='tight')
print(f'💾 Guardado: {ruta_global.name}')
plt.show()
plt.close(fig)